# Perbandingan GA, PSO, ACO, dan ABC pada Penempatan Sensor dan Relai Karhutla

Notebook ini mencari lokasi sensor dan relai agar sel dengan kerawanan tinggi terpantau sebanyak mungkin, dengan biaya total tidak melebihi anggaran. Data yang dipakai adalah data simulasi.

**Cara pakai**
1. Jalankan sel pertama, lalu unggah tiga berkas: `kerawanan.csv`, `valid.csv`, dan `parameter.json`.
2. Pilih menu Runtime lalu Run all.
3. Waktu jalan sekitar beberapa menit. Untuk uji cepat, kecilkan `ULANG` dan `ITERASI` pada bagian pengaturan.

**Cara kerja singkat**
- Setiap algoritma mengusulkan penempatan sensor dan relai.
- Fungsi `evaluasi` menghitung cakupan dari penempatan tersebut. Cakupan adalah jumlah skor kerawanan sel yang terpantau dibagi total skor kerawanan seluruh sel. Nilainya 0 sampai 1, makin besar makin baik.
- Keempat algoritma memakai fungsi `evaluasi` yang sama dan jumlah evaluasi yang hampir sama, sehingga perbandingannya adil.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BERKAS = ["kerawanan.csv", "valid.csv", "parameter.json"]
if not all(os.path.exists(b) for b in BERKAS):
    from google.colab import files
    files.upload()   # pilih ketiga berkas sekaligus

## 1. Membaca data dan parameter

In [ ]:
P = json.load(open("parameter.json"))
N_GRID = P["ukuran_grid"][0]            # grid N_GRID x N_GRID
R_SENSOR = P["radius_sensor_r"]         # radius deteksi sensor
R_RELAI = P["radius_relai_R"]           # jarak maksimum sensor ke relai
KAPASITAS = P["kapasitas_relai_N"]      # jumlah sensor maksimum per relai
JARAK_MIN = P["jarak_minimum_antarsensor"]
BIAYA_SENSOR = P["biaya_sensor_cs"]
BIAYA_RELAI = P["biaya_relai_cr"]
ANGGARAN = P["anggaran_B"]

skor = np.loadtxt("kerawanan.csv", delimiter=",").ravel()                # skor tiap sel
valid = np.loadtxt("valid.csv", delimiter=",").ravel().astype(bool)      # True = boleh dipasangi alat

# Setiap sel punya satu nomor (id) dari 0 sampai N_GRID * N_GRID - 1, dihitung baris demi baris
id_sel = np.arange(N_GRID * N_GRID)
baris, kolom = id_sel // N_GRID, id_sel % N_GRID
jarak = np.hypot(baris[:, None] - baris[None, :], kolom[:, None] - kolom[None, :])   # jarak antar sel
dipantau = jarak <= R_SENSOR       # dipantau[i] = sel yang terpantau oleh sensor di sel i
print("Sel valid:", valid.sum(), "dari", len(valid))
print("Parameter: r =", R_SENSOR, ", R =", R_RELAI, ", N =", KAPASITAS, ", anggaran =", ANGGARAN)

## 2. Pengaturan algoritma

Jumlah evaluasi per proses sekitar `POPULASI * ITERASI`, yaitu 4000. GA, PSO, dan ABC memakai sedikit evaluasi tambahan untuk kondisi awal dan lebah penjelajah, dengan selisih kurang dari 3%.

In [ ]:
POPULASI = 40      # jumlah individu (GA), partikel (PSO), semut (ACO), atau lebah (ABC) per iterasi
ITERASI = 100
ULANG = 10         # jumlah pengulangan per metode dan anggaran (disarankan 30 untuk laporan akhir)

PC, PM = 0.9, 0.05            # GA: peluang crossover, peluang mutasi per alat
W, C1, C2, VMAKS = 0.7, 1.5, 1.5, 0.2   # PSO: inersia, koefisien pbest, koefisien gbest, batas kecepatan
ALPHA, BETA, RHO = 1.0, 2.0, 0.1        # ACO: bobot pheromone, bobot heuristik, laju penguapan
BATAS_ABC, MR_ABC = 20, 0.05            # ABC: batas gagal sebelum sumber ditinggalkan, peluang tiap angka diubah

## 3. Fungsi evaluasi

Fungsi ini menilai satu penempatan. Aturannya:
1. Alat di sel air dibuang. Relai ganda di sel yang sama dihitung satu.
2. Sensor yang berada di sel relai dibuang.
3. Sensor yang terlalu dekat dengan sensor sebelumnya (kurang dari `JARAK_MIN`) dibuang. Sensor ganda juga terbuang di langkah ini.
4. Setiap sensor terhubung ke relai terdekat dalam jarak `R_RELAI`. Satu relai melayani paling banyak `KAPASITAS` sensor, sisanya tidak terhubung.
5. Hanya sel di sekitar sensor yang terhubung yang dihitung sebagai terpantau.

In [ ]:
def evaluasi(relai, sensor, detail=False):
    """Menghitung cakupan (0 sampai 1). relai dan sensor berisi id sel."""
    relai = np.unique(relai[valid[relai]])                          # aturan 1
    sensor = sensor[valid[sensor] & ~np.isin(sensor, relai)]        # aturan 1 dan 2
    terlalu_dekat = np.tril(jarak[np.ix_(sensor, sensor)] < JARAK_MIN, k=-1).any(axis=1)
    sensor = sensor[~terlalu_dekat]                                 # aturan 3

    if len(relai) == 0 or len(sensor) == 0:
        terhubung = np.zeros(len(sensor), dtype=bool)
        return (0.0, relai, sensor, terhubung) if detail else 0.0

    d = jarak[np.ix_(sensor, relai)]                                # jarak tiap sensor ke tiap relai
    d[d > R_RELAI] = np.inf
    terdekat = d.argmin(axis=1)
    terhubung = np.isfinite(d.min(axis=1))                          # aturan 4
    for j in range(len(relai)):
        anggota = np.where(terhubung & (terdekat == j))[0]
        terhubung[anggota[KAPASITAS:]] = False                      # relai penuh

    sel_terpantau = dipantau[sensor[terhubung]].any(axis=0)         # aturan 5
    cakupan = skor[sel_terpantau].sum() / skor.sum()
    return (cakupan, relai, sensor, terhubung) if detail else cakupan

## 4. Representasi solusi (untuk GA, PSO, dan ABC)

Satu solusi adalah satu vektor angka dengan susunan berikut:
- Elemen pertama adalah `m`, jumlah relai.
- Setelah itu berurutan posisi (baris, kolom) untuk semua slot relai, lalu semua slot sensor.
- Hanya `m` slot relai pertama yang dipakai. Jumlah sensor mengikuti sisa anggaran, yaitu `(B - m * biaya relai) / biaya sensor`.

Dengan susunan ini biaya tidak pernah melebihi anggaran, dan algoritma sekaligus mencari jumlah relai, jumlah sensor, dan lokasinya.

In [ ]:
def ukuran(B):
    maks_relai = B // BIAYA_RELAI
    maks_sensor = (B - BIAYA_RELAI) // BIAYA_SENSOR      # jumlah sensor jika hanya ada 1 relai
    return maks_relai, maks_sensor

def baca_solusi(x, B):
    """Mengubah vektor x menjadi (id relai, id sensor)."""
    maks_relai, _ = ukuran(B)
    m = int(np.clip(np.rint(x[0]), 1, maks_relai))
    jumlah_sensor = (B - m * BIAYA_RELAI) // BIAYA_SENSOR
    posisi = np.clip(np.rint(x[1:]).astype(int), 0, N_GRID - 1).reshape(-1, 2)
    id_alat = posisi[:, 0] * N_GRID + posisi[:, 1]
    return id_alat[:m], id_alat[maks_relai:maks_relai + jumlah_sensor]

def nilai(x, B):
    return evaluasi(*baca_solusi(x, B))

def individu_acak(rng, B):
    maks_relai, maks_sensor = ukuran(B)
    x = rng.integers(0, N_GRID, 1 + 2 * (maks_relai + maks_sensor))
    x[0] = rng.integers(1, maks_relai + 1)
    return x

## 5. Genetic Algorithm (GA)

Langkah per generasi: individu terbaik dibawa langsung (elitisme), lalu individu lain dibuat dari dua induk hasil seleksi turnamen, digabung dengan crossover, dan diubah sedikit dengan mutasi.

In [ ]:
def pilih_induk(pop, skor_pop, rng):
    calon = rng.integers(0, len(pop), 3)                     # turnamen: ambil 3 individu, pilih terbaik
    return pop[calon[skor_pop[calon].argmax()]]

def ga(B, seed):
    rng = np.random.default_rng(seed)
    maks_relai, _ = ukuran(B)
    pop = np.array([individu_acak(rng, B) for _ in range(POPULASI)])
    skor_pop = np.array([nilai(x, B) for x in pop])
    riwayat, jejak = [], []
    for it in range(ITERASI):
        baru = [pop[skor_pop.argmax()].copy()]               # elitisme
        while len(baru) < POPULASI:
            a, b = pilih_induk(pop, skor_pop, rng), pilih_induk(pop, skor_pop, rng)
            anak = a.copy()
            if rng.random() < PC:                            # crossover: tiap alat diambil dari induk a atau b
                dari_b = np.repeat(rng.random((len(a) - 1) // 2) < 0.5, 2)
                anak[1:][dari_b] = b[1:][dari_b]
                if rng.random() < 0.5:
                    anak[0] = b[0]
            if rng.random() < PM:                            # mutasi jumlah relai
                anak[0] = np.clip(anak[0] + rng.integers(-2, 3), 1, maks_relai)
            posisi = anak[1:].reshape(-1, 2)                 # mutasi posisi alat
            geser = rng.random(len(posisi)) < PM             # geser sedikit
            posisi[geser] = np.clip(posisi[geser] + rng.integers(-2, 3, (geser.sum(), 2)), 0, N_GRID - 1)
            lompat = rng.random(len(posisi)) < PM / 5        # pindah ke sel acak
            posisi[lompat] = rng.integers(0, N_GRID, (lompat.sum(), 2))
            baru.append(anak)
        pop = np.array(baru)
        skor_pop = np.array([nilai(x, B) for x in pop])
        riwayat.append(skor_pop.max())
        jejak.append(baca_solusi(pop[skor_pop.argmax()], B))      # penempatan terbaik iterasi ini (untuk video)
    return (skor_pop.max(), *baca_solusi(pop[skor_pop.argmax()], B), riwayat, jejak)

## 6. Particle Swarm Optimization (PSO)

Setiap partikel adalah vektor angka kontinu dengan susunan yang sama seperti di atas. Angka dibulatkan menjadi sel saat dievaluasi. Kecepatan dipengaruhi oleh kecepatan sebelumnya, posisi terbaik partikel itu sendiri (pbest), dan posisi terbaik seluruh kelompok (gbest).

In [ ]:
def pso(B, seed):
    rng = np.random.default_rng(seed)
    maks_relai, maks_sensor = ukuran(B)
    D = 1 + 2 * (maks_relai + maks_sensor)
    bawah = np.zeros(D); bawah[0] = 1
    atas = np.full(D, N_GRID - 1.0); atas[0] = maks_relai
    x = rng.uniform(bawah, atas, (POPULASI, D))
    v = np.zeros((POPULASI, D))
    skor_x = np.array([nilai(p, B) for p in x])
    pbest, skor_pbest = x.copy(), skor_x.copy()
    g = skor_pbest.argmax()
    gbest, skor_gbest = pbest[g].copy(), skor_pbest[g]
    riwayat, jejak = [], []
    for it in range(ITERASI):
        r1, r2 = rng.random((2, POPULASI, D))
        v = W * v + C1 * r1 * (pbest - x) + C2 * r2 * (gbest - x)
        v = np.clip(v, -VMAKS * (atas - bawah), VMAKS * (atas - bawah))
        x = np.clip(x + v, bawah, atas)
        skor_x = np.array([nilai(p, B) for p in x])
        lebih_baik = skor_x > skor_pbest
        pbest[lebih_baik], skor_pbest[lebih_baik] = x[lebih_baik], skor_x[lebih_baik]
        g = skor_pbest.argmax()
        if skor_pbest[g] > skor_gbest:
            gbest, skor_gbest = pbest[g].copy(), skor_pbest[g]
        riwayat.append(skor_gbest)
        jejak.append(baca_solusi(gbest, B))
    return (skor_gbest, *baca_solusi(gbest, B), riwayat, jejak)

## 7. Ant Colony Optimization (ACO)

ACO bekerja dengan pilihan sel, jadi solusinya dibangun langkah demi langkah:
1. Semut memilih jumlah relai `m`, lalu memilih `m` sel relai, lalu memilih sensor satu per satu sampai sisa anggaran habis.
2. Peluang suatu pilihan bergantung pada pheromone (pengalaman iterasi sebelumnya) dan heuristik (skor kerawanan di sekitar sel).
3. Setelah semua semut selesai, pheromone menguap sebagian, lalu diperkuat pada pilihan milik semut terbaik iterasi itu.

Catatan: heuristik memakai pengetahuan tentang masalah, dan itu memang bagian dari rancangan ACO. GA dan PSO tidak memakainya.

In [ ]:
id_valid = id_sel[valid]                                           # ACO bekerja pada sel valid saja
dekat_valid = jarak[np.ix_(id_valid, id_valid)] < JARAK_MIN        # pasangan sel valid yang terlalu dekat
bobot_sensor = (dipantau @ skor)[id_valid]                         # total skor di sekitar sel (radius sensor)
bobot_relai = ((jarak <= R_RELAI) @ skor)[id_valid]                # total skor di sekitar sel (radius relai)
eta_sensor = 0.01 + bobot_sensor / bobot_sensor.max()              # heuristik, 0.01 agar tidak ada peluang nol
eta_relai = 0.01 + bobot_relai / bobot_relai.max()

def aco(B, seed):
    rng = np.random.default_rng(seed)
    maks_relai, _ = ukuran(B)
    nv = len(id_valid)
    tau_m, tau_relai, tau_sensor = np.ones(maks_relai), np.ones(nv), np.ones(nv)   # pheromone awal
    terbaik, riwayat, jejak = (-1, None, None), [], []
    for it in range(ITERASI):
        hasil = []
        for semut in range(POPULASI):
            m = rng.choice(maks_relai, p=tau_m / tau_m.sum()) + 1
            p = tau_relai ** ALPHA * eta_relai ** BETA
            idx_relai = rng.choice(nv, size=m, replace=False, p=p / p.sum())
            jumlah_sensor = (B - m * BIAYA_RELAI) // BIAYA_SENSOR
            p = tau_sensor ** ALPHA * eta_sensor ** BETA
            p[idx_relai] = 0                                 # sel relai tidak dipakai sensor
            idx_sensor = []
            for k in range(jumlah_sensor):
                if p.sum() == 0:
                    break
                c = rng.choice(nv, p=p / p.sum())
                idx_sensor.append(c)
                p[dekat_valid[c]] = 0                        # sel sekitar tidak dipilih lagi (jarak minimum)
            idx_sensor = np.array(idx_sensor, dtype=int)
            f = evaluasi(id_valid[idx_relai], id_valid[idx_sensor])
            hasil.append((f, m, idx_relai, idx_sensor))
        f, m, idx_relai, idx_sensor = max(hasil, key=lambda h: h[0])    # semut terbaik iterasi ini
        for tau in (tau_m, tau_relai, tau_sensor):
            tau *= (1 - RHO)                                 # penguapan
            np.maximum(tau, 0.01, out=tau)
        tau_m[m - 1] += f                                    # penguatan
        tau_relai[idx_relai] += f
        tau_sensor[idx_sensor] += f
        if f > terbaik[0]:
            terbaik = (f, id_valid[idx_relai], id_valid[idx_sensor])
        riwayat.append(terbaik[0])
        jejak.append((terbaik[1], terbaik[2]))
    return (*terbaik, riwayat, jejak)

## 8. Artificial Bee Colony (ABC)

Koloni terdiri dari 40 lebah, yaitu 20 lebah pekerja dan 20 lebah pengamat. Setiap sumber makanan adalah satu solusi dengan susunan vektor yang sama seperti GA dan PSO, sehingga ada 20 sumber makanan. Setiap siklus terdiri dari tiga fase:
1. Lebah pekerja: tiap pekerja mencoba memperbaiki sumber miliknya dengan mengubah beberapa angka berdasarkan sumber lain (peluang `MR_ABC` per angka, minimal satu). Perubahan hanya diterima jika cakupan naik.
2. Lebah pengamat: memilih sumber berdasarkan cakupannya (sumber yang lebih baik lebih sering dipilih), lalu mencoba memperbaikinya dengan cara yang sama.
3. Lebah penjelajah: sumber yang gagal diperbaiki lebih dari `BATAS_ABC` kali ditinggalkan dan diganti dengan solusi acak baru.

Satu siklus memakai sekitar 40 evaluasi, sama dengan satu iterasi pada algoritma lain.

In [ ]:
def abc(B, seed):
    rng = np.random.default_rng(seed)
    maks_relai, maks_sensor = ukuran(B)
    D = 1 + 2 * (maks_relai + maks_sensor)
    bawah = np.zeros(D); bawah[0] = 1
    atas = np.full(D, N_GRID - 1.0); atas[0] = maks_relai
    SN = POPULASI // 2                                    # jumlah sumber makanan (sama dengan jumlah pekerja dan pengamat)
    x = rng.uniform(bawah, atas, (SN, D))                 # sumber makanan, satu baris = satu solusi
    f = np.array([nilai(p, B) for p in x])
    gagal = np.zeros(SN, dtype=int)                       # jumlah percobaan berturut-turut yang tidak memperbaiki sumber
    g = f.argmax()
    gbest, f_gbest = x[g].copy(), f[g]
    riwayat, jejak = [], []

    def coba(i):                                          # cari tetangga sumber i, ganti jika lebih baik
        k = rng.choice(np.delete(np.arange(SN), i))       # sumber lain sebagai acuan
        ubah = rng.random(D) < MR_ABC                     # angka-angka yang diubah
        ubah[rng.integers(D)] = True                      # minimal satu angka berubah
        baru = x[i].copy()
        baru[ubah] = np.clip(x[i, ubah] + rng.uniform(-1, 1, ubah.sum()) * (x[i, ubah] - x[k, ubah]), bawah[ubah], atas[ubah])
        f_baru = nilai(baru, B)
        if f_baru > f[i]:
            x[i], f[i], gagal[i] = baru, f_baru, 0
        else:
            gagal[i] += 1

    for it in range(ITERASI):
        for i in range(SN):                               # fase lebah pekerja
            coba(i)
        peluang = (f + 1e-9) / (f + 1e-9).sum()           # sumber yang lebih baik lebih sering dipilih
        for i in rng.choice(SN, size=SN, p=peluang):      # fase lebah pengamat
            coba(i)
        for i in np.where(gagal > BATAS_ABC)[0]:          # fase lebah penjelajah
            x[i] = rng.uniform(bawah, atas)
            f[i], gagal[i] = nilai(x[i], B), 0
        g = f.argmax()
        if f[g] > f_gbest:
            gbest, f_gbest = x[g].copy(), f[g]
        riwayat.append(f_gbest)
        jejak.append(baca_solusi(gbest, B))
    return (f_gbest, *baca_solusi(gbest, B), riwayat, jejak)

## 9. Pembanding: pencarian acak

Metode ini hanya mengambil solusi acak dengan jumlah evaluasi yang sama. Jika suatu algoritma tidak mengalahkan pencarian acak, algoritma itu tidak banyak membantu pada pengaturan yang dipakai.

In [ ]:
def acak(B, seed):
    rng = np.random.default_rng(seed)
    terbaik, riwayat, jejak = (-1, None), [], []
    for it in range(ITERASI):
        for k in range(POPULASI):
            x = individu_acak(rng, B)
            f = nilai(x, B)
            if f > terbaik[0]:
                terbaik = (f, x)
        riwayat.append(terbaik[0])
        jejak.append(baca_solusi(terbaik[1], B))
    return (terbaik[0], *baca_solusi(terbaik[1], B), riwayat, jejak)

## 10. Menjalankan semua metode

Setiap metode dijalankan `ULANG` kali pada setiap anggaran. Seed berbeda untuk tiap pengulangan, sehingga hasilnya dapat diulang persis.

In [ ]:
METODE = {"Acak": acak, "GA": ga, "PSO": pso, "ACO": aco, "ABC": abc}
ALGORITMA = ["GA", "PSO", "ACO", "ABC"]        # empat algoritma yang dibandingkan (pencarian acak hanya pembanding)
baris_hasil, riwayat, terbaik = [], {}, {}

for B in ANGGARAN:
    for nama, fungsi in METODE.items():
        for seed in range(ULANG):
            mulai = time.time()
            f, relai, sensor, riw, jejak = fungsi(B, seed)
            waktu = time.time() - mulai
            _, relai_ok, sensor_ok, terhubung = evaluasi(relai, sensor, detail=True)
            baris_hasil.append(dict(
                anggaran=B, metode=nama, seed=seed, cakupan=f,
                relai=len(relai_ok), sensor=len(sensor_ok), terhubung=int(terhubung.sum()),
                biaya=len(relai_ok) * BIAYA_RELAI + len(sensor_ok) * BIAYA_SENSOR, waktu=waktu))
            riwayat.setdefault((B, nama), []).append(riw)
            if f > terbaik.get((B, nama), (-1,))[0]:
                terbaik[(B, nama)] = (f, relai, sensor, jejak)
    print("Selesai anggaran", B)

df = pd.DataFrame(baris_hasil)
assert (df.biaya <= df.anggaran).all(), "Ada solusi yang melebihi anggaran"

## 11. Hasil

Tabel berikut memuat rata-rata dan simpangan baku cakupan, cakupan terbaik, rata-rata jumlah relai dan sensor yang terpasang, rata-rata sensor yang terhubung, biaya terpakai, dan waktu per proses.

In [ ]:
ringkas = df.groupby(["anggaran", "metode"]).agg(
    rata2=("cakupan", "mean"), std=("cakupan", "std"), terbaik=("cakupan", "max"),
    relai=("relai", "mean"), sensor=("sensor", "mean"), terhubung=("terhubung", "mean"),
    biaya=("biaya", "mean"), waktu_detik=("waktu", "mean")).round(3)
ringkas

In [ ]:
# Grafik 1: cakupan terhadap anggaran (garis = rata-rata, batang = simpangan baku)
plt.figure(figsize=(6.5, 4))
for nama in METODE:
    d = df[df.metode == nama].groupby("anggaran").cakupan
    plt.errorbar(ANGGARAN, d.mean(), yerr=d.std(), marker="o", capsize=3, label=nama)
plt.xlabel("Anggaran"); plt.ylabel("Cakupan"); plt.title("Cakupan terhadap anggaran")
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Grafik 2: kurva konvergensi untuk setiap anggaran (rata-rata cakupan terbaik tiap iterasi)
fig, ax = plt.subplots(1, len(ANGGARAN), figsize=(4.2 * len(ANGGARAN), 3.8), sharey=True)
for a, B in zip(ax, ANGGARAN):
    for nama in METODE:
        a.plot(np.mean(riwayat[(B, nama)], axis=0), label=nama)
    a.set_title(f"Anggaran {B}")
    a.set_xlabel("Iterasi"); a.grid(alpha=0.3)
ax[0].set_ylabel("Cakupan terbaik"); ax[0].legend()
plt.tight_layout(); plt.show()

## 12. Peta penempatan terbaik

Peta menampilkan hasil terbaik dari seluruh pengulangan, satu gambar untuk setiap anggaran. Area putih adalah air. Warna merah menunjukkan skor kerawanan. Hijau transparan menandai area yang terpantau oleh sensor yang terhubung ke relai (radius sensor), sehingga warna merah di bawahnya tetap terlihat.

In [ ]:
from matplotlib.colors import ListedColormap

peta = np.ma.masked_where(~valid, skor).reshape(N_GRID, N_GRID)

def gambar_peta(a, judul, relai, sensor):                # menggambar satu peta penempatan pada sumbu a
    cakupan, relai, sensor, terhubung = evaluasi(relai, sensor, detail=True)
    a.clear()
    a.imshow(peta, cmap="OrRd", vmin=0, vmax=1)
    terpantau = dipantau[sensor[terhubung]].any(axis=0).reshape(N_GRID, N_GRID)   # radius sensor yang terhubung
    a.imshow(np.ma.masked_where(~terpantau, terpantau), cmap=ListedColormap(["#00b050"]), alpha=0.3)   # overlay hijau transparan
    a.scatter([], [], marker="s", s=60, c="#00b050", alpha=0.5, label="area terpantau")
    a.scatter(kolom[sensor[terhubung]], baris[sensor[terhubung]], c="black", s=18, label="sensor terhubung")
    a.scatter(kolom[sensor[~terhubung]], baris[sensor[~terhubung]], c="gray", marker="x", s=22, label="sensor tidak terhubung")
    a.scatter(kolom[relai], baris[relai], c="blue", marker="^", s=80, label="relai")
    a.set_title(f"{judul}, cakupan {cakupan:.3f}, {len(relai)} relai, {len(sensor)} sensor")

for B in ANGGARAN:                                       # satu gambar untuk setiap anggaran
    fig, ax = plt.subplots(1, len(ALGORITMA), figsize=(5 * len(ALGORITMA), 5.3))
    for a, nama in zip(ax, ALGORITMA):
        gambar_peta(a, nama, *terbaik[(B, nama)][1:3])
    fig.suptitle(f"Penempatan terbaik, anggaran {B}")
    fig.legend(*ax[0].get_legend_handles_labels(), loc="lower center", ncol=4)
    plt.tight_layout(rect=(0, 0.09, 1, 0.95)); plt.show()

## 13. Video perubahan penempatan tiap iterasi

Video memperlihatkan penempatan terbaik yang sudah ditemukan GA, PSO, ACO, dan ABC pada setiap iterasi, dari iterasi 1 sampai iterasi terakhir. Dibuat satu video untuk setiap anggaran, dari pengulangan dengan hasil terbaik pada tiap metode. Berkas `video_anggaran_30.mp4` dan seterusnya tersimpan di folder kerja dan dapat diunduh dari panel Files. Pembuatan video memerlukan ffmpeg, yang sudah tersedia di Colab.

Bagian ini memakai hasil dari bagian 10 (menjalankan semua metode). Jika Anda mengganti notebook pada sesi Colab yang sudah berjalan, pilih Runtime lalu Run all agar semua sel dijalankan ulang dari awal.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import Video, display

# Pengaman: hasil dari versi lama notebook tidak menyimpan data per iterasi
if len(terbaik[(ANGGARAN[0], "GA")]) < 4:
    raise RuntimeError("Hasil lama terdeteksi. Pilih Runtime lalu Run all agar semua sel dijalankan ulang dari awal.")

FPS = 8     # jumlah iterasi yang ditampilkan per detik video

def buat_video(B):
    jejak = {nama: terbaik[(B, nama)][3] for nama in ALGORITMA}
    fig, ax = plt.subplots(1, len(ALGORITMA), figsize=(5 * len(ALGORITMA), 5.3))

    def gambar_iterasi(i):
        for a, nama in zip(ax, jejak):
            gambar_peta(a, f"{nama}, iterasi {i + 1}", *jejak[nama][i])

    gambar_iterasi(0)
    fig.suptitle(f"Penempatan terbaik pada tiap iterasi, anggaran {B}")
    fig.legend(*ax[0].get_legend_handles_labels(), loc="lower center", ncol=4)
    plt.tight_layout(rect=(0, 0.09, 1, 0.95))
    berkas = f"video_anggaran_{B}.mp4"
    FuncAnimation(fig, gambar_iterasi, frames=ITERASI).save(berkas, writer="ffmpeg", fps=FPS, dpi=80)
    plt.close(fig)
    return berkas

for B in ANGGARAN:
    print("Membuat video anggaran", B)
    display(Video(buat_video(B), embed=True, width=900))

## 14. Uji statistik (opsional)

Uji Mann-Whitney membandingkan dua kelompok hasil pengulangan. Nilai p di bawah 0,05 berarti perbedaan rata-rata kedua metode kecil kemungkinannya terjadi karena kebetulan.

In [ ]:
from itertools import combinations
from scipy.stats import mannwhitneyu

for B in ANGGARAN:
    for a, b in combinations(ALGORITMA, 2):              # semua pasangan dari empat algoritma
        xa = df[(df.anggaran == B) & (df.metode == a)].cakupan
        xb = df[(df.anggaran == B) & (df.metode == b)].cakupan
        p = mannwhitneyu(xa, xb).pvalue
        print(f"Anggaran {B}: {a} vs {b}, p = {p:.4f}", "(berbeda nyata)" if p < 0.05 else "(tidak berbeda nyata)")